# DE2 - Assignment 3: Graphs or Clustering
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** B - Smart Mobility IoT

**Path chosen:** Clustering (KMeans + BisectingKMeans)

**Dataset:** `smart_mobility_dataset.csv` - 5 000 IoT sensor readings, NYC metro area, March 2024

In [1]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
import time, pathlib

DE2_SPARK_DRIVER_HOST  = os.environ.get("DE2_SPARK_DRIVER_HOST",  "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)


def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows browser):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")


spark = (
    SparkSession.builder
    .appName("de2-assignment3")
    .master("local[*]")
    .config("spark.driver.host",        DE2_SPARK_DRIVER_HOST)
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS)
    .config("spark.ui.bindAddress",     DE2_SPARK_BIND_ADDRESS)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

show_spark_ui(spark)

Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows browser): http://localhost:4040


## Step 1 - Data and Features

Load `smart_mobility_dataset.csv`

In [2]:
import csv, statistics, datetime
import pathlib, time, json
import requests
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans, BisectingKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

OUT_DIR   = pathlib.Path("outputs/lab3")
PROOF_DIR = pathlib.Path("proof")
for d in [OUT_DIR, PROOF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CSV_PATH = "smart_mobility_dataset.csv"

df_raw = spark.read.csv(CSV_PATH, header=True, inferSchema=True)

df_raw = df_raw.withColumnRenamed("Road_Occupancy_%", "Road_Occupancy_pct")

print(f"Loaded {df_raw.count()} rows, {len(df_raw.columns)} columns")
df_raw.printSchema()
df_raw.show(5, truncate=False)

Loaded 5000 rows, 15 columns
root
 |-- Timestamp: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Vehicle_Count: integer (nullable = true)
 |-- Traffic_Speed_kmh: double (nullable = true)
 |-- Road_Occupancy_pct: double (nullable = true)
 |-- Traffic_Light_State: string (nullable = true)
 |-- Weather_Condition: string (nullable = true)
 |-- Accident_Report: integer (nullable = true)
 |-- Sentiment_Score: double (nullable = true)
 |-- Ride_Sharing_Demand: integer (nullable = true)
 |-- Parking_Availability: integer (nullable = true)
 |-- Emission_Levels_g_km: double (nullable = true)
 |-- Energy_Consumption_L_h: double (nullable = true)
 |-- Traffic_Condition: string (nullable = true)

+-------------------+------------------+------------------+-------------+------------------+------------------+-------------------+-----------------+---------------+--------------------+-------------------+--------------------+-------------

In [3]:
FEATURE_COLS = [
    "Vehicle_Count",
    "Traffic_Speed_kmh",
    "Road_Occupancy_pct",
    "Accident_Report",
    "Emission_Levels_g_km",
    "Energy_Consumption_L_h",
    "Ride_Sharing_Demand",
    "Parking_Availability",
]

df_clean = df_raw.dropna(subset=FEATURE_COLS)
print(f"Rows after null check: {df_clean.count()}")

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="raw_features",
)
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True,
)
prep_pipeline = Pipeline(stages=[assembler, scaler])
prep_model = prep_pipeline.fit(df_clean)

df_features = prep_model.transform(df_clean).drop("raw_features")

df_features.cache()
n_rows = df_features.count()
print(f"Feature DataFrame cached: {n_rows} rows")
print(f"Number of input partitions: {df_features.rdd.getNumPartitions()}")
df_features.select("features").show(3, truncate=True)

Rows after null check: 5000
Feature DataFrame cached: 5000 rows
Number of input partitions: 1
+--------------------+
|            features|
+--------------------+
|[0.61155120135381...|
|[0.57563309826483...|
|[1.17426814974783...|
+--------------------+
only showing top 3 rows


## Step 2 - Iterative Algorithm

In [4]:
evaluator = ClusteringEvaluator(
    predictionCol="prediction",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean",
)

k_sweep_results = []

print("KMeans k-sweep (k = 2 to 6) ...")
print(f"{'k':>3}  {'silhouette':>10}  {'inertia':>12}  {'elapsed_s':>9}")
print("-" * 42)

for k in range(2, 7):
    t0 = time.time()
    km = KMeans(
        featuresCol="features",
        predictionCol="prediction",
        k=k,
        seed=42,
        maxIter=20,
    )
    model_k   = km.fit(df_features)
    df_pred_k = model_k.transform(df_features)
    sil       = evaluator.evaluate(df_pred_k)
    elapsed   = time.time() - t0
    inertia   = model_k.summary.trainingCost

    k_sweep_results.append({
        "k":          k,
        "silhouette": round(sil, 4),
        "inertia":    round(inertia, 2),
        "elapsed_s":  round(elapsed, 2),
    })
    print(f"{k:>3}  {sil:>10.4f}  {inertia:>12.1f}  {elapsed:>9.2f}")

best_entry = max(k_sweep_results, key=lambda r: r["silhouette"])
BEST_K     = best_entry["k"]
print(f"\nBest k = {BEST_K}  (silhouette = {best_entry['silhouette']})")

KMeans k-sweep (k = 2 to 6) ...
  k  silhouette       inertia  elapsed_s
------------------------------------------
  2      0.1691       36222.3       3.04
  3      0.1433       34035.2       1.78
  4      0.1886       29590.7       1.49
  5      0.1550       30601.4       1.45
  6      0.1919       26520.8       1.48

Best k = 6  (silhouette = 0.1919)


In [5]:
km_best    = KMeans(featuresCol="features", predictionCol="prediction",
                   k=BEST_K, seed=42, maxIter=20)
model_best = km_best.fit(df_features)
df_pred_best = model_best.transform(df_features)

print(f"Cluster sizes (k = {BEST_K}):")
df_pred_best.groupBy("prediction").count().orderBy("prediction").show()

print("Cluster centers (standardized feature space):")
print(f"  Columns: {FEATURE_COLS}")
for i, center in enumerate(model_best.clusterCenters()):
    print(f"  Cluster {i}: {[round(v, 3) for v in center]}")

Cluster sizes (k = 6):
+----------+-----+
|prediction|count|
+----------+-----+
|         0|  912|
|         1|  881|
|         2|  892|
|         3|  951|
|         4|  480|
|         5|  884|
+----------+-----+

Cluster centers (standardized feature space):
  Columns: ['Vehicle_Count', 'Traffic_Speed_kmh', 'Road_Occupancy_pct', 'Accident_Report', 'Emission_Levels_g_km', 'Energy_Consumption_L_h', 'Ride_Sharing_Demand', 'Parking_Availability']
  Cluster 0: [np.float64(0.77), np.float64(0.517), np.float64(0.079), np.float64(-0.326), np.float64(0.984), np.float64(0.092), np.float64(-0.162), np.float64(0.209)]
  Cluster 1: [np.float64(0.862), np.float64(-0.089), np.float64(0.009), np.float64(-0.326), np.float64(-0.862), np.float64(-0.628), np.float64(-0.202), np.float64(-0.071)]
  Cluster 2: [np.float64(-0.924), np.float64(-0.087), np.float64(-0.188), np.float64(-0.326), np.float64(0.276), np.float64(-0.691), np.float64(-0.598), np.float64(-0.071)]
  Cluster 3: [np.float64(-0.135), np.flo

In [6]:
t0 = time.time()
bkm  = BisectingKMeans(featuresCol="features", predictionCol="prediction", k=BEST_K, seed=42, maxIter=20)
model_bkm  = bkm.fit(df_features)
df_pred_bkm = model_bkm.transform(df_features)

num_clusters = df_pred_bkm.select("prediction").distinct().count()

if num_clusters > 1:
    sil_bkm = evaluator.evaluate(df_pred_bkm)
else:
    print(f"Warning : BisectingKMeans generated only {num_clusters} cluster.")
    sil_bkm = 0.0

elapsed_bkm = time.time() - t0

print(f"\nAlgorithm comparison at k = {BEST_K}:")
print(f"  KMeans         silhouette = {best_entry['silhouette']:.4f}")
print(f"  BisectingKMeans silhouette = {sil_bkm:.4f}  ({elapsed_bkm:.2f}s)")


Algorithm comparison at k = 6:
  KMeans         silhouette = 0.1919
  BisectingKMeans silhouette = 0.0000  (3.87s)


Analyse des performances : KMeans vs BisectingKMeans
1. Observations des résultats

KMeans (k=6) a réussi à forcer la création de 6 sous-groupes, mais obtient un score de Silhouette très faible d'environ 0.1919.

BisectingKMeans (k=6), de son côté, s'est arrêté prématurément et n'a conservé qu'un seul et unique cluster global, générant de facto un score de Silhouette nul.

2. Interprétation algorithmique

Les données ayant été correctement normalisées en amont (StandardScaler), ce contraste montre la différence fondamentale d'approche entre ces deux modèles :

KMeans utilise une approche "brute" (géométrique) : il pose ses centroïdes et partitionne artificiellement l'espace de données, même s'il n'y a pas de véritables frontières naturelles. Le faible score de silhouette prouve d'ailleurs que ces clusters se chevauchent fortement.

BisectingKMeans utilise une approche hiérarchique divisive, c'est à dire qu'à chaque itération il évalue s'il est statistiquement pertinent de couper un cluster en deux (réduction de la variance). S'il juge que la séparation n'apporte pas de gain significatif, il ne divise pas.

3. Conclusion sur le jeu de données IoT

Le comportement "capricieux" du BisectingKMeans n'est donc pas une erreur, mais une information analytique précieuse. Il démontre que les mesures des capteurs IoT de mobilité (Vehicle_Count, Traffic_Speed, etc.) forment ici un nuage de données extrêmement dense et continu. Il n'existe pas de typologies ou de comportements de trafic naturellement disjoints (ou très séparés) selon la distance euclidienne classique dans cet espace de caractéristiques.

## Step 3 - Partitioning Experiment

Two strategies are compared at the best k:

| Strategy | shuffle.partitions | Input partitions | Description |
|---|---|---|---|
| A - default | 4 | inherited from CSV reader | baseline, no repartition |
| B - repartitioned | 8 | 8 (explicit repartition) | even distribution, more parallelism |

Shuffle bytes are read from the Spark REST API (`/api/v1/applications/.../stages`).

In [7]:
def get_cumulative_shuffle_write():
    """Return cumulative shuffle-write bytes from the Spark REST API."""
    try:
        resp   = requests.get("http://localhost:4040/api/v1/applications", timeout=2)
        app_id = resp.json()[0]["id"]
        stages = requests.get(
            f"http://localhost:4040/api/v1/applications/{app_id}/stages", timeout=2
        ).json()
        return sum(s.get("shuffleWriteBytes", 0) for s in stages)
    except Exception:
        return -1

In [8]:
# Strategy A: default partitioning (shuffle.partitions = 4)
spark.conf.set("spark.sql.shuffle.partitions", "4")

sw_before_A = get_cumulative_shuffle_write()
t0 = time.time()

km_A    = KMeans(featuresCol="features", predictionCol="prediction",
                 k=BEST_K, seed=42, maxIter=20)
model_A = km_A.fit(df_features)
pred_A  = model_A.transform(df_features)
sil_A   = evaluator.evaluate(pred_A)

elapsed_A = time.time() - t0
sw_A      = get_cumulative_shuffle_write() - max(sw_before_A, 0)

print("Strategy A - default (shuffle.partitions=4, no repartition)")
print(f"  Input partitions : {df_features.rdd.getNumPartitions()}")
print(f"  Silhouette       : {sil_A:.4f}")
print(f"  Elapsed          : {elapsed_A:.2f}s")
print(f"  Shuffle write    : {sw_A:,} bytes  (-1 means UI not reachable)")

Strategy A - default (shuffle.partitions=4, no repartition)
  Input partitions : 1
  Silhouette       : 0.1919
  Elapsed          : 1.28s
  Shuffle write    : 18,049 bytes  (-1 means UI not reachable)


In [9]:
# Strategy B: explicit repartition + doubled shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "8")

df_features_B = df_features.repartition(8)
df_features_B.cache()
df_features_B.count()   # materialise cache

sw_before_B = get_cumulative_shuffle_write()
t0 = time.time()

km_B    = KMeans(featuresCol="features", predictionCol="prediction",
                 k=BEST_K, seed=42, maxIter=20)
model_B = km_B.fit(df_features_B)
pred_B  = model_B.transform(df_features_B)
sil_B   = evaluator.evaluate(pred_B)

elapsed_B = time.time() - t0
sw_B      = get_cumulative_shuffle_write() - max(sw_before_B, 0)

print("Strategy B - repartitioned (shuffle.partitions=8, repartition(8))")
print(f"  Input partitions : {df_features_B.rdd.getNumPartitions()}")
print(f"  Silhouette       : {sil_B:.4f}")
print(f"  Elapsed          : {elapsed_B:.2f}s")
print(f"  Shuffle write    : {sw_B:,} bytes")

delta_sil  = (sil_B - sil_A)
delta_time = elapsed_A - elapsed_B
print(f"\nDelta silhouette  : {delta_sil:+.4f}")
print(f"Delta elapsed (A-B): {delta_time:+.2f}s  (positive = B is faster)")

Strategy B - repartitioned (shuffle.partitions=8, repartition(8))
  Input partitions : 8
  Silhouette       : 0.1912
  Elapsed          : 3.39s
  Shuffle write    : 376,668 bytes

Delta silhouette  : -0.0007
Delta elapsed (A-B): -2.11s  (positive = B is faster)


## Step 4 - Convergence and Stability Analysis

In [10]:
SEEDS = [0, 7, 13, 42, 99]
sil_scores   = []
inertia_list = []

print(f"Seed stability: KMeans k={BEST_K} over seeds {SEEDS} ...")
print(f"{'seed':>6}  {'silhouette':>10}  {'inertia':>12}")
print("-" * 34)

for seed in SEEDS:
    km_s    = KMeans(featuresCol="features", predictionCol="prediction",
                     k=BEST_K, seed=seed, maxIter=20)
    model_s = km_s.fit(df_features)
    df_s    = model_s.transform(df_features)
    sil_s   = evaluator.evaluate(df_s)
    iner_s  = model_s.summary.trainingCost

    sil_scores.append(sil_s)
    inertia_list.append(iner_s)
    print(f"{seed:>6}  {sil_s:>10.4f}  {iner_s:>12.1f}")

sil_mean = statistics.mean(sil_scores)
sil_std  = statistics.stdev(sil_scores)
cv_pct   = (sil_std / sil_mean) * 100 if sil_mean != 0 else float("inf")

print(f"\nSilhouette mean  : {sil_mean:.4f}")
print(f"Silhouette std   : {sil_std:.4f}")
print(f"CV (std/mean)    : {cv_pct:.2f}%")
stability = "Stable (CV < 2%)" if cv_pct < 2 else "Moderate variance" if cv_pct < 5 else "High variance"
print(f"Assessment       : {stability}")

Seed stability: KMeans k=6 over seeds [0, 7, 13, 42, 99] ...
  seed  silhouette       inertia
----------------------------------
     0      0.1890       26638.6
     7      0.1924       26569.0
    13      0.1914       26543.3
    42      0.1919       26520.8
    99      0.1910       26531.6

Silhouette mean  : 0.1911
Silhouette std   : 0.0013
CV (std/mean)    : 0.69%
Assessment       : Stable (CV < 2%)


## Step 5 - Evidence and Metrics

Save execution plans, write `lab3_metrics_log.csv`, persist cluster assignments,
and print the Spark UI URL for screenshot capture.

In [11]:
# Execution plans: before repartition (Strategy A) and after (Strategy B)
ts = datetime.datetime.utcnow().isoformat() + " UTC"

plan_before = df_features._jdf.queryExecution().executedPlan().toString()
plan_after  = df_features_B._jdf.queryExecution().executedPlan().toString()
plan_iter   = pred_A._jdf.queryExecution().executedPlan().toString()

(PROOF_DIR / "plan_before.txt").write_text(
    f"Generated: {ts}\nStrategy A (default partitioning)\n\n" + plan_before,
    encoding="utf-8",
)
(PROOF_DIR / "plan_after.txt").write_text(
    f"Generated: {ts}\nStrategy B (repartition to 8)\n\n" + plan_after,
    encoding="utf-8",
)
(PROOF_DIR / "plan_iteration.txt").write_text(
    f"Generated: {ts}\nKMeans prediction plan (k={BEST_K}, Strategy A)\n\n" + plan_iter,
    encoding="utf-8",
)
print("Plans written to proof/")

# Persist cluster assignments (best-k, strategy A)
out_assignments = OUT_DIR / "cluster_assignments"
(
    pred_A
    .select("prediction", *FEATURE_COLS)
    .write.mode("overwrite")
    .parquet(str(out_assignments))
)
print(f"Cluster assignments written to {out_assignments}")

Plans written to proof/
Cluster assignments written to outputs\lab3\cluster_assignments


In [12]:
METRICS_CSV = pathlib.Path("lab3_metrics_log.csv")
FIELDNAMES  = [
    "experiment", "algorithm", "k", "strategy",
    "shuffle_partitions", "num_partitions",
    "silhouette", "elapsed_s", "inertia",
    "shuffle_write_bytes", "seed", "notes",
]

rows = []

for r in k_sweep_results:
    rows.append({
        "experiment": "k_sweep", "algorithm": "KMeans",
        "k": r["k"], "strategy": "default",
        "shuffle_partitions": 4,
        "num_partitions": df_features.rdd.getNumPartitions(),
        "silhouette": r["silhouette"], "elapsed_s": r["elapsed_s"],
        "inertia": r["inertia"], "shuffle_write_bytes": "",
        "seed": 42, "notes": "",
    })

rows.append({
    "experiment": "algo_compare", "algorithm": "BisectingKMeans",
    "k": BEST_K, "strategy": "default",
    "shuffle_partitions": 4,
    "num_partitions": df_features.rdd.getNumPartitions(),
    "silhouette": round(sil_bkm, 4), "elapsed_s": round(elapsed_bkm, 2),
    "inertia": "", "shuffle_write_bytes": "",
    "seed": 42, "notes": "divisive hierarchy",
})

rows.append({
    "experiment": "partitioning", "algorithm": "KMeans",
    "k": BEST_K, "strategy": "A_default",
    "shuffle_partitions": 4,
    "num_partitions": df_features.rdd.getNumPartitions(),
    "silhouette": round(sil_A, 4), "elapsed_s": round(elapsed_A, 2),
    "inertia": "", "shuffle_write_bytes": sw_A,
    "seed": 42, "notes": "baseline - no repartition",
})
rows.append({
    "experiment": "partitioning", "algorithm": "KMeans",
    "k": BEST_K, "strategy": "B_repartition_8",
    "shuffle_partitions": 8,
    "num_partitions": 8,
    "silhouette": round(sil_B, 4), "elapsed_s": round(elapsed_B, 2),
    "inertia": "", "shuffle_write_bytes": sw_B,
    "seed": 42, "notes": "repartition(8) + shuffle.partitions=8",
})

for seed, sil in zip(SEEDS, sil_scores):
    rows.append({
        "experiment": "seed_stability", "algorithm": "KMeans",
        "k": BEST_K, "strategy": "default",
        "shuffle_partitions": 4,
        "num_partitions": df_features.rdd.getNumPartitions(),
        "silhouette": round(sil, 4), "elapsed_s": "",
        "inertia": "", "shuffle_write_bytes": "",
        "seed": seed, "notes": "",
    })

with open(METRICS_CSV, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(rows)

print(f"Metrics CSV written: {METRICS_CSV}  ({len(rows)} rows)")
print()
print("Contents:")
print(",".join(FIELDNAMES))
for r in rows:
    print(",".join(str(r[f]) for f in FIELDNAMES))

Metrics CSV written: lab3_metrics_log.csv  (13 rows)

Contents:
experiment,algorithm,k,strategy,shuffle_partitions,num_partitions,silhouette,elapsed_s,inertia,shuffle_write_bytes,seed,notes
k_sweep,KMeans,2,default,4,1,0.1691,3.04,36222.35,,42,
k_sweep,KMeans,3,default,4,1,0.1433,1.78,34035.19,,42,
k_sweep,KMeans,4,default,4,1,0.1886,1.49,29590.72,,42,
k_sweep,KMeans,5,default,4,1,0.155,1.45,30601.43,,42,
k_sweep,KMeans,6,default,4,1,0.1919,1.48,26520.79,,42,
algo_compare,BisectingKMeans,6,default,4,1,0.0,3.87,,,42,divisive hierarchy
partitioning,KMeans,6,A_default,4,1,0.1919,1.28,,18049,42,baseline - no repartition
partitioning,KMeans,6,B_repartition_8,8,8,0.1912,3.39,,376668,42,repartition(8) + shuffle.partitions=8
seed_stability,KMeans,6,default,4,1,0.189,,,,0,
seed_stability,KMeans,6,default,4,1,0.1924,,,,7,
seed_stability,KMeans,6,default,4,1,0.1914,,,,13,
seed_stability,KMeans,6,default,4,1,0.1919,,,,42,
seed_stability,KMeans,6,default,4,1,0.191,,,,99,


In [13]:
print("SELF-CHECK")

print(f"Path             : Clustering")
print(f"Track            : B - Smart Mobility IoT")
print(f"Feature vectors  : {df_features.count()} rows, {len(FEATURE_COLS)} features")
print(f"K-sweep range    : k = 2..6  (5 iterations with metrics)")
print(f"Best k           : {BEST_K}")
print(f"Silhouette best  : {best_entry['silhouette']}")
print(f"Strategies       : A (default) vs B (repartition_8)")
print(f"Seed stability   : {len(SEEDS)} seeds, mean={sil_mean:.4f} std={sil_std:.4f}")
print(f"Metrics CSV      : {METRICS_CSV}  ({len(rows)} rows)")
for fname in ["plan_before.txt", "plan_after.txt", "plan_iteration.txt"]:
    exists = (PROOF_DIR / fname).exists()
    print(f"  proof/{fname} : {'OK' if exists else 'MISSING'}")
cluster_files = list((OUT_DIR / 'cluster_assignments').rglob('*.parquet')) if (OUT_DIR / 'cluster_assignments').exists() else []
print(f"  outputs/lab3/cluster_assignments : {len(cluster_files)} parquet file(s)")

ui_url = spark.sparkContext.uiWebUrl or "http://localhost:4040"
print(f"\nSpark UI for screenshots:")
print(f"  {ui_url}/stages/")
print(f"  {ui_url}/SQL/")

SELF-CHECK
Path             : Clustering
Track            : B - Smart Mobility IoT
Feature vectors  : 5000 rows, 8 features
K-sweep range    : k = 2..6  (5 iterations with metrics)
Best k           : 6
Silhouette best  : 0.1919
Strategies       : A (default) vs B (repartition_8)
Seed stability   : 5 seeds, mean=0.1911 std=0.0013
Metrics CSV      : lab3_metrics_log.csv  (13 rows)
  proof/plan_before.txt : OK
  proof/plan_after.txt : OK
  proof/plan_iteration.txt : OK
  outputs/lab3/cluster_assignments : 1 parquet file(s)

Spark UI for screenshots:
  http://127.0.0.1:4040/stages/
  http://127.0.0.1:4040/SQL/


In [14]:
spark.stop()
print("Done.")

Done.
